<a href="https://colab.research.google.com/github/SoulaimakH/phishing-detection/blob/main/phishing_detection_pfe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Install dependencies
!pip install dnspython python-whois tldextract ipwhois

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.8 MB/s eta 0:00:00


In [ ]:
import dns.resolver
import socket
import whois
import tldextract
import datetime
import re
from urllib.parse import urlparse
import math
from collections import Counter
import string

# -----------------------------
# Helper functions
# -----------------------------

# Entropy calculation
def entropy(s):
    p, lns = Counter(s), len(s)
    return -sum(count/lns * math.log2(count/lns) for count in p.values())

# N-grams
def ngrams(s, n=2):
    return [s[i:i+n] for i in range(len(s)-n+1)]

# Numeric percentage
def numeric_percentage(s):
    digits = sum(c.isdigit() for c in s)
    return digits / len(s) * 100 if len(s) > 0 else 0

# Longest word (letters only)
def longest_word(s):
    words = re.findall(r'[a-zA-Z]+', s)
    return max(words, key=len) if words else ""

# Obfuscation checks
def check_hex(s): return int(bool(re.search(r'%[0-9a-fA-F]{2}', s)))
def check_at(s): return int('@' in s)
def check_decimal(s): return int(bool(re.search(r'\b\d{2,}\b', s)))
def check_octal(s): return int(bool(re.search(r'\\[0-7]{1,3}', s)))

# Typosquatting placeholder (optional, can use Levenshtein distance)
def typos_similar(domain, popular_domains=['google.com','apple.com']):
    # return list of tuples (domain, similarity score)
    return [(d, 0) for d in popular_domains]

# -----------------------------
# Main feature extraction function
# -----------------------------
def extract_features(domain_name):
    features = {}

    # Normalize domain
    domain_name = domain_name.lower().strip()

    # ---------------------
    # DNS / Network info
    # ---------------------
    try:
        answers = dns.resolver.resolve(domain_name, 'A')
        ip = answers[0].to_text()
        ttl = answers.rrset.ttl
    except:
        ip = None
        ttl = 0

    features['IP'] = ip
    features['TTL'] = ttl

    # Country & ASN lookup (optional, requires geoip2 or ipwhois)
    try:
        from ipwhois import IPWhois
        obj = IPWhois(ip)
        res = obj.lookup_rdap()
        features['ASN'] = res.get('asn', None)
        features['Country'] = res.get('asn_country_code', None)
    except:
        features['ASN'] = None
        features['Country'] = None

    # Name server count
    try:
        ns = dns.resolver.resolve(domain_name, 'NS')
        features['Name_Server_Count'] = len(ns)
    except:
        features['Name_Server_Count'] = 0

    # ---------------------
    # WHOIS info
    # ---------------------
    try:
        w = whois.whois(domain_name)
        features['Registrant_Name'] = w.get('name', None)
        features['Organization'] = w.get('org', None)
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        features['Creation_Date_Time'] = creation_date
        features['Domain_Age'] = (datetime.datetime.now() - creation_date).days if creation_date else 0
    except:
        features['Registrant_Name'] = None
        features['Organization'] = None
        features['Creation_Date_Time'] = None
        features['Domain_Age'] = 0

    # ---------------------
    # Domain structure
    # ---------------------
    ext = tldextract.extract(domain_name)
    features['sld'] = ext.domain
    features['tld'] = ext.suffix
    features['subdomain'] = ext.subdomain
    features['Domain_Name'] = domain_name
    features['len'] = len(domain_name)
    features['longest_word'] = longest_word(domain_name)

    # ---------------------
    # Text / NLP features
    # ---------------------
    features['entropy'] = entropy(domain_name)
    features['1gram'] = list(domain_name)
    features['2gram'] = ngrams(domain_name, 2)
    features['3gram'] = ngrams(domain_name, 3)
    features['char_distribution'] = dict(Counter(domain_name))
    features['numeric_percentage'] = numeric_percentage(domain_name)
    features['typos'] = typos_similar(domain_name)

    # ---------------------
    # Obfuscation features
    # ---------------------
    features['hex_8'] = check_hex(domain_name)
    features['hex_32'] = check_hex(domain_name)  # placeholder, can refine
    features['dec_8'] = check_decimal(domain_name)
    features['dec_32'] = check_decimal(domain_name)
    features['oc_8'] = check_octal(domain_name)
    features['oc_32'] = check_octal(domain_name)
    features['obfuscate_at_sign'] = check_at(domain_name)
    features['puny_coded'] = int(domain_name.startswith('xn--'))

    return features

# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    domain = "www.ksylitol.com"
    features = extract_features(domain)
    for k,v in features.items():
        print(f"{k}: {v}")

IP: 91.228.196.192
TTL: 14400
ASN: 41079 198414
Country: PL
Name_Server_Count: 0
Registrant_Name: None
Organization: None
Creation_Date_Time: None
Domain_Age: 0
sld: ksylitol
tld: com
subdomain: www
Domain_Name: www.ksylitol.com
len: 16
longest_word: ksylitol
entropy: 3.327819531114783
1gram: ['w', 'w', 'w', '.', 'k', 's', 'y', 'l', 'i', 't', 'o', 'l', '.', 'c', 'o', 'm']
2gram: ['ww', 'ww', 'w.', '.k', 'ks', 'sy', 'yl', 'li', 'it', 'to', 'ol', 'l.', '.c', 'co', 'om']
3gram: ['www', 'ww.', 'w.k', '.ks', 'ksy', 'syl', 'yli', 'lit', 'ito', 'tol', 'ol.', 'l.c', '.co', 'com']
char_distribution: {'w': 3, '.': 2, 'k': 1, 's': 1, 'y': 1, 'l': 2, 'i': 1, 't': 1, 'o': 2, 'c': 1, 'm': 1}
numeric_percentage: 0.0
typos: [('google.com', 0), ('apple.com', 0)]
hex_8: 0
hex_32: 0
dec_8: 0
dec_32: 0
oc_8: 0
oc_32: 0
obfuscate_at_sign: 0
puny_coded: 0


In [3]:
pip install dnspython ipwhois numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.6 MB/s eta 0:00:00


In [12]:
import dns.resolver
import socket
import numpy as np
from ipwhois import IPWhois

def get_dns_features(domain):
    """
    Extract DNS statistical features (F15-F21) for a domain.
    Returns a dictionary.
    """
    try:
        # Resolve A records
        answers = dns.resolver.resolve(domain, 'A')
    except Exception as e:
        print(f"Error resolving domain {domain}: {e}")
        return None

    # IPs
    ips = [rdata.address for rdata in answers]

    # TTL: use answers.rrset.ttl
    ttls = [answers.rrset.ttl] * len(ips)  # same TTL for all records in this answer

    countries = []
    asns = []
    reverse_domains = []

    for ip in ips:
        # ASN & Country lookup
        try:
            obj = IPWhois(ip)
            res = obj.lookup_rdap()
            asns.append(res.get('asn', 'NA'))
            countries.append(res.get('network', {}).get('country', 'NA'))
        except Exception as e:
            asns.append('NA')
            countries.append('NA')

        # Reverse DNS lookup
        try:
            rev = socket.gethostbyaddr(ip)
            reverse_domains.append(rev[0])
        except:
            continue

    # Compute features
    F15_unique_country = len(set([c for c in countries if c != 'NA']))
    F16_unique_asn = len(set([a for a in asns if a != 'NA']))
    F17_unique_ttl = len(set(ttls))
    F18_unique_ip = len(set(ips))
    F19_unique_domain = len(set(reverse_domains))
    F20_ttl_mean = np.mean(ttls) if ttls else 0
    F21_ttl_variance = np.var(ttls) if ttls else 0

    features = {
        'F15_unique_country': F15_unique_country,
        'F16_unique_asn': F16_unique_asn,
        'F17_unique_ttl': F17_unique_ttl,
        'F18_unique_ip': F18_unique_ip,
        'F19_unique_domain': F19_unique_domain,
        'F20_ttl_mean': F20_ttl_mean,
        'F21_ttl_variance': F21_ttl_variance
    }

    return features

# Example usage
if __name__ == "__main__":
    domain = "a1professionals.net"
    features = get_dns_features(domain)
    print(f"DNS Features for {domain}:")
    print(features)

DNS Features for a1professionals.net:
{'F15_unique_country': 1, 'F16_unique_asn': 1, 'F17_unique_ttl': 1, 'F18_unique_ip': 2, 'F19_unique_domain': 0, 'F20_ttl_mean': np.float64(300.0), 'F21_ttl_variance': np.float64(0.0)}


In [13]:
pip install python-whois requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.4 MB/s eta 0:00:00


In [16]:
import whois
import datetime
import requests

def get_third_party_features(domain):
    features = {}

    # F22 – Domain name
    features['F22_domain'] = domain

    # WHOIS lookup
    try:
        w = whois.whois(domain)

        # F23 – Registrar
        features['F23_registrar'] = w.registrar if w.registrar else 'NA'

        # F24 – Registrant name
        features['F24_registrant_name'] = w.name if w.name else 'NA'

        # F25 – Creation date/time
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        features['F25_creation_date'] = creation_date if creation_date else 'NA'

        # F26 – Emails
        emails = w.emails
        if isinstance(emails, list):
            emails = ','.join(emails)
        features['F26_emails'] = emails if emails else 'NA'

        # F27 – Domain age in days
        if creation_date and creation_date != 'NA':
            # handle timezone-aware dates
            if creation_date.tzinfo is not None:
                creation_date = creation_date.replace(tzinfo=None)  # convert to naive
            age = (datetime.datetime.now() - creation_date).days
            features['F27_domain_age'] = age
        else:
            features['F27_domain_age'] = 'NA'

        # F28 – Organization
        features['F28_organization'] = w.org if w.org else 'NA'

        # F29 – State
        features['F29_state'] = w.state if w.state else 'NA'

        # F30 – Country
        features['F30_country'] = w.country if w.country else 'NA'

        # F31 – Name server count
        if w.name_servers:
            if isinstance(w.name_servers, list):
                features['F31_name_server_count'] = len(w.name_servers)
            else:
                features['F31_name_server_count'] = 1
        else:
            features['F31_name_server_count'] = 0

    except Exception as e:
        print(f"Whois error for {domain}: {e}")
        # fill NA for all
        for i in range(23, 32):
            features[f'F{i}'] = 'NA'

    # F32 – Alexa rank
    try:
        url = f"http://data.alexa.com/data?cli=10&url={domain}"
        r = requests.get(url)
        if r.status_code == 200 and 'POPULARITY' in r.text:
            import xml.etree.ElementTree as ET
            root = ET.fromstring(r.content)
            pop = root.find('.//POPULARITY')
            rank = int(pop.attrib['TEXT']) if pop is not None else -1
            features['F32_alexa_rank'] = rank
        else:
            features['F32_alexa_rank'] = -1
    except:
        features['F32_alexa_rank'] = -1

    return features

# Example usage
if __name__ == "__main__":
    domain = "facebook.com"
    features = get_third_party_features(domain)
    for k, v in features.items():
        print(k, v)

F22_domain facebook.com
F23_registrar RegistrarSafe, LLC
F24_registrant_name Domain Admin
F25_creation_date 1997-03-29 05:00:00+00:00
F26_emails abusecomplaints@registrarsafe.com,domain@fb.com
F27_domain_age 10598
F28_organization Meta Platforms, Inc.
F29_state CA
F30_country US
F31_name_server_count 4
F32_alexa_rank -1


In [18]:
import dns.resolver
import socket
import whois
import tldextract
import datetime
import re
import math
from collections import Counter
import requests

# -----------------------------
# Helper functions
# -----------------------------

# Entropy calculation
def entropy(s):
    p, lns = Counter(s), len(s)
    return -sum(count/lns * math.log2(count/lns) for count in p.values()) if lns > 0 else 0

# N-grams
def ngrams(s, n=2):
    return [s[i:i+n] for i in range(len(s)-n+1)] if len(s) >= n else []

# Numeric percentage
def numeric_percentage(s):
    digits = sum(c.isdigit() for c in s)
    return digits / len(s) * 100 if len(s) > 0 else 0

# Longest word (letters only)
def longest_word(s):
    words = re.findall(r'[a-zA-Z]+', s)
    return max(words, key=len) if words else ""

# Obfuscation checks
def check_hex(s): return int(bool(re.search(r'%[0-9a-fA-F]{2}', s))) #Good for detecting URL encoding (percent-encoding) (	Contains %6c, which is hex for the letter 'l'.)
def check_at(s): return int('@' in s)#A classic phishing indicator. The @ symbol in a URL causes many browsers to ignore everything to the left of it, redirecting the user to the domain on the right.
def check_decimal(s): return int(bool(re.search(r'\b\d{2,}\b', s))) #Useful for finding IP-based URLs or long numeric strings which are uncommon in legitimate, brand-named domains => (http://192.168.1)
def check_octal(s): return int(bool(re.search(r'\\[0-7]{1,3}', s)))  #Detects escape sequences. While rare in standard URLs, these are sometimes used in obfuscated scripts or payloads => (http://site.com\154\157\147\151\156)
def check_punycode(s): return int(s.startswith('xn--'))#Essential for spotting IDN Homograph attacks (e.g., using a Cyrillic 'а' instead of a Latin 'a') => (http://xn--80ak6aa92e.com)

# Placeholder for typosquatting
def typos_similar(domain, popular_domains=['google.com','apple.com']):
    # Could implement Levenshtein distance here
    return [(d, 0) for d in popular_domains]

# -----------------------------
# Main feature extraction
# -----------------------------
def extract_domain_features(domain_name):
    features = {}
    domain_name = domain_name.lower().strip()

    # ---------------------
    # Domain structure / lexical
    # ---------------------
    ext = tldextract.extract(domain_name)
    features['Domain_Name'] = domain_name
    features['SLD'] = ext.domain
    features['TLD'] = ext.suffix
    features['Subdomain'] = ext.subdomain
    features['Length'] = len(domain_name)
    features['Longest_Word'] = longest_word(domain_name)
    features['Entropy'] = entropy(domain_name)
    features['1gram'] = list(domain_name)
    features['2gram'] = ngrams(domain_name, 2)
    features['3gram'] = ngrams(domain_name, 3)
    features['Char_Distribution'] = dict(Counter(domain_name))
    features['Numeric_Percentage'] = numeric_percentage(domain_name)
    features['Typos'] = typos_similar(domain_name)

    # Obfuscation
    features['Hex'] = check_hex(domain_name)
    features['Decimal'] = check_decimal(domain_name)
    features['Octal'] = check_octal(domain_name)
    features['At_Sign'] = check_at(domain_name)
    features['Punycode'] = check_punycode(domain_name)

    # ---------------------
    # DNS / Network features
    # ---------------------
    try:
        answers = dns.resolver.resolve(domain_name, 'A')
        ips = [r.to_text() for r in answers]
        ttl = answers.rrset.ttl
    except:
        ips = []
        ttl = 0
    features['IP_Addresses'] = ips
    features['TTL'] = ttl

    try:
        ns = dns.resolver.resolve(domain_name, 'NS')
        features['Name_Server_Count'] = len(ns)
    except:
        features['Name_Server_Count'] = 0

    try:
        from ipwhois import IPWhois
        if ips:
            obj = IPWhois(ips[0])
            res = obj.lookup_rdap()
            features['ASN'] = res.get('asn', None)
            features['Country'] = res.get('asn_country_code', None)
        else:
            features['ASN'] = None
            features['Country'] = None
    except:
        features['ASN'] = None
        features['Country'] = None

    # ---------------------
    # WHOIS / Third-party features
    # ---------------------
    try:
        w = whois.whois(domain_name)
        features['Registrar'] = w.registrar if w.registrar else 'NA'
        features['Registrant_Name'] = w.name if w.name else 'NA'
        features['Organization'] = w.org if w.org else 'NA'
        features['State'] = w.state if w.state else 'NA'
        features['Country_WHOIS'] = w.country if w.country else 'NA'

        # Emails
        emails = w.emails
        if isinstance(emails, list):
            emails = ','.join(emails)
        features['Emails'] = emails if emails else 'NA'

        # Creation date
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        features['Creation_Date'] = creation_date if creation_date else 'NA'

        # Domain age
        if creation_date and creation_date != 'NA':
            if creation_date.tzinfo:
                creation_date = creation_date.replace(tzinfo=None)
            features['Domain_Age'] = (datetime.datetime.now() - creation_date).days
        else:
            features['Domain_Age'] = 'NA'
    except:
        for key in ['Registrar','Registrant_Name','Organization','State','Country_WHOIS','Emails','Creation_Date','Domain_Age']:
            features[key] = 'NA'

    # Alexa rank
    try:
        url = f"http://data.alexa.com/data?cli=10&url={domain_name}"
        r = requests.get(url)
        if r.status_code == 200 and 'POPULARITY' in r.text:
            import xml.etree.ElementTree as ET
            root = ET.fromstring(r.content)
            pop = root.find('.//POPULARITY')
            rank = int(pop.attrib['TEXT']) if pop is not None else -1
            features['Alexa_Rank'] = rank
        else:
            features['Alexa_Rank'] = -1
    except:
        features['Alexa_Rank'] = -1

    return features

# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    domain = "facebook.com"
    features = extract_domain_features(domain)
    for k, v in features.items():
        print(f"{k}: {v}")

Domain_Name: facebook.com
SLD: facebook
TLD: com
Subdomain: 
Length: 12
Longest_Word: facebook
Entropy: 3.0220552088742005
1gram: ['f', 'a', 'c', 'e', 'b', 'o', 'o', 'k', '.', 'c', 'o', 'm']
2gram: ['fa', 'ac', 'ce', 'eb', 'bo', 'oo', 'ok', 'k.', '.c', 'co', 'om']
3gram: ['fac', 'ace', 'ceb', 'ebo', 'boo', 'ook', 'ok.', 'k.c', '.co', 'com']
Char_Distribution: {'f': 1, 'a': 1, 'c': 2, 'e': 1, 'b': 1, 'o': 3, 'k': 1, '.': 1, 'm': 1}
Numeric_Percentage: 0.0
Typos: [('google.com', 0), ('apple.com', 0)]
Hex: 0
Decimal: 0
Octal: 0
At_Sign: 0
Punycode: 0
IP_Addresses: ['31.13.66.35']
TTL: 2
Name_Server_Count: 4
ASN: 32934
Country: IE
Registrar: RegistrarSafe, LLC
Registrant_Name: Domain Admin
Organization: Meta Platforms, Inc.
State: CA
Country_WHOIS: US
Emails: abusecomplaints@registrarsafe.com,domain@fb.com
Creation_Date: 1997-03-29 05:00:00+00:00
Domain_Age: 10598
Alexa_Rank: -1
